# Web Scraping Project: Books to Scrape

Scrapes book data (title, price, rating, availability, link) from [books.toscrape.com](https://books.toscrape.com), a site built for practicing web scraping, then cleans the data and saves it to CSV.

**Steps:** fetch pages → parse HTML → clean data → analyze → export

**Tools:** `requests`, `BeautifulSoup`, `pandas`

In [ ]:
# !pip install requests beautifulsoup4 pandas matplotlib
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"
HEADERS = {"User-Agent": "Mozilla/5.0 (learning project; web-scraping-demo)"}
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

## 1. Fetch a page

In [ ]:
def fetch(url, retries=3, delay=1.0):
    """GET a URL with simple retry logic. Returns HTML text or None."""
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=10)
            if resp.status_code == 404:
                return None  # past the last page
            resp.raise_for_status()
            resp.encoding = "utf-8"
            return resp.text
        except requests.RequestException as e:
            print(f"Attempt {attempt} failed for {url}: {e}")
            time.sleep(delay * attempt)
    return None

## 2. Parse the HTML

In [ ]:
def parse_books(html, page_url):
    """Extract book records from one listing page."""
    soup = BeautifulSoup(html, "html.parser")
    books = []
    for card in soup.select("article.product_pod"):
        link = card.h3.a
        rating_word = card.select_one("p.star-rating")["class"][1]
        books.append({
            "title": link["title"],
            "price": card.select_one("p.price_color").text,
            "rating": RATING_MAP.get(rating_word),
            "in_stock": "In stock" in card.select_one("p.availability").text,
            "url": urljoin(page_url, link["href"]),
        })
    return books

## 3. Scrape all pages

A short delay between requests keeps the scraper polite to the server.

In [ ]:
def scrape_all(max_pages=50, delay=0.5):
    all_books = []
    for page in range(1, max_pages + 1):
        url = BASE_URL.format(page)
        html = fetch(url)
        if html is None:
            print(f"Stopped at page {page} (no more pages).")
            break
        books = parse_books(html, url)
        all_books.extend(books)
        print(f"Page {page}: {len(books)} books")
        time.sleep(delay)
    return all_books

raw = scrape_all()
print(f"\nTotal scraped: {len(raw)} books")

## 4. Clean the data

In [ ]:
df = pd.DataFrame(raw)
# '£51.77' -> 51.77 (strip every non-numeric character except the dot)
df["price"] = df["price"].str.replace(r"[^0-9.]", "", regex=True).astype(float)
df = df.drop_duplicates(subset="url").reset_index(drop=True)
df.head()

In [ ]:
df.info()
df.describe()

## 5. Quick analysis

In [ ]:
print("Average price: £%.2f" % df["price"].mean())
print("\nBooks per rating:")
print(df["rating"].value_counts().sort_index())
print("\nTop 5 most expensive:")
df.nlargest(5, "price")[["title", "price", "rating"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["price"].plot.hist(bins=20, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set(title="Price distribution", xlabel="Price (£)")
df.groupby("rating")["price"].mean().plot.bar(ax=axes[1], color="darkorange")
axes[1].set(title="Average price by rating", xlabel="Rating (stars)", ylabel="Price (£)")
plt.tight_layout()
plt.show()

## 6. Export

In [ ]:
df.to_csv("books.csv", index=False)
print(f"Saved {len(df)} rows to books.csv")

## Notes
- Only scrape sites you have permission to scrape; check `robots.txt` and the terms of service first.
- Possible extensions: scrape each book's detail page (description, UPC), store results in SQLite, or schedule the scraper.